In [ ]:
# Cell 2 — imports + config
import time
import os
import math
import csv
import requests
from datetime import datetime, timezone
import folium
from folium.features import DivIcon
import webbrowser

URL = "https://broker.fiware.urbanplatform.portodigital.pt/v2/entities"
PARAMS = {"q": "vehicleType==bus", "limit": 1000}

# Monitor multiple routes (defaults, can be overridden by the UI)
NEEDLE_ROUTES = [
    "stcp:route:200",
    "stcp:route:205",
    "stcp:route:305",
    "stcp:route:400",
    "stcp:route:500",
    "stcp:route:503",
    "stcp:route:600",
    "stcp:route:601",
    "stcp:route:700",
    "stcp:route:704",
    "stcp:route:803",
    "stcp:route:804",
    "stcp:route:903",
    "stcp:route:907",
]
# Monitor both directions (sentidos)
NEEDLE_SENTIDOS = ["stcp:sentido:0", "stcp:sentido:1"]

# If True, force using simple ASCII arrows (<, >, ^, v). Otherwise prefer Unicode arrows if available.
USE_ASCII_ARROWS = False

OUT_MAP = "map.html"
REFRESH_SECONDS = 30
ROUTES_GTFS_PATH = "routes.txt"
# In-memory history of previous point-sets (keeps up to 5 previous refreshes)
history = []

# Cell 3 — helpers
def get_timestamp(entity: dict):
    for k in ["observationDateTime", "dateObserved", "timestamp"]:
        if k in entity:
            v = entity[k]
            if isinstance(v, dict) and "value" in v:
                return v["value"]
            return v
    return None

def get_heading(entity: dict):
    h = entity.get("heading")
    if isinstance(h, dict) and "value" in h:
        return h["value"]
    return h

def format_time(ts):
    if not ts:
        return "N/A"
    try:
        if isinstance(ts, str):
            s = ts
        elif isinstance(ts, dict) and "value" in ts:
            s = ts["value"]
        else:
            s = str(ts)
        if s.endswith('Z'):
            s = s.replace('Z', '+00:00')
        dt = datetime.fromisoformat(s)
        return dt.strftime('%H:%M:%S')
    except Exception:
        return str(ts)

def parse_time_to_dt(ts):
    """Parse timestamp and return a timezone-aware datetime object.
    Returns None if parsing fails.
    """
    if not ts:
        return None
    try:
        if isinstance(ts, str):
            s = ts
        elif isinstance(ts, dict) and "value" in ts:
            s = ts["value"]
        else:
            s = str(ts)
        if s.endswith('Z'):
            s = s.replace('Z', '+00:00')
        return datetime.fromisoformat(s)
    except Exception:
        return None

def heading_to_direction(h):
    # normalize
    h = float(h) % 360
    if (h >= 337.5) or (h < 22.5):
        return 'N'
    if 22.5 <= h < 67.5:
        return 'NE'
    if 67.5 <= h < 112.5:
        return 'E'
    if 112.5 <= h < 157.5:
        return 'SE'
    if 157.5 <= h < 202.5:
        return 'S'
    if 202.5 <= h < 247.5:
        return 'SW'
    if 247.5 <= h < 292.5:
        return 'W'
    return 'NW'

def direction_glyph(dir_code, use_ascii=False):
    unicode_map = {
        'N': '↑', 'NE': '↗', 'E': '→', 'SE': '↘',
        'S': '↓', 'SW': '↙', 'W': '←', 'NW': '↖',
    }
    if not use_ascii:
        return unicode_map.get(dir_code, '')
    # ASCII fallback: use cardinal glyphs for diagonals
    if dir_code == 'N':
        return '^'
    if dir_code == 'S':
        return 'v'
    if dir_code in ('E', 'NE', 'SE'):
        return '>'
    return '<'


def meters_to_latlon_offset(lat, lon, meters_north, meters_east):
    # Approximate conversions
    lat_offset = meters_north / 111111
    lon_offset = meters_east / (111111 * math.cos(math.radians(lat)))
    return lat + lat_offset, lon + lon_offset


def fetch_entities():
    r = requests.get(URL, params=PARAMS, headers={"Accept": "application/json"}, timeout=30)
    r.raise_for_status()
    return r.json()


def color_for_route_and_sentido(route_line: str, sentido: str, color_type: str) -> str:
    # Returns a hex color for the given route_line and sentido strings
    # sentido expected to be 'stcp:sentido:0' or 'stcp:sentido:1'
    s_idx = 0 if sentido and sentido.endswith(':0') else 1
    try:
        r = int(route_line) if route_line is not None else None
    except Exception:
        r = None

    if color_type == 'text':
        if r is None:
            return '#FFFFFF'
        if 500 <= r <= 599:
            return '#000000'
        else:
            return '#FFFFFF'
    else:
        if r is None:
            return '#555555' if s_idx == 0 else '#000000'
        if 1 <= r <= 99:
            return '#AB803D'
        if 100 <= r <= 499:
            return ['#06579E', '#268FFF'][s_idx]
        if 500 <= r <= 599:
            return ['#8F7C00', '#E1C403'][s_idx]
        if 600 <= r <= 699:
            return ['#00800B', '#00C911'][s_idx]
        if 700 <= r <= 799:
            return ['#B00000', '#FF0000'][s_idx]
        if 800 <= r <= 899:
            return ['#7302A7', '#B51AFD'][s_idx]
        if 900 <= r <= 999:
            return ['#B05601', '#F28118'][s_idx]
        return ['#000000', '#555555'][s_idx]


def extract_vehicle_id(full_id):
    """Extract the vehicle ID from the full entity ID (e.g., '3416' from 'urn:ngsi-ld:Vehicle:porto:stcp:bus:3416')"""
    if not full_id:
        return None
    parts = full_id.split(':')
    return parts[-1] if parts else None


def get_nr_viagem(annotations):
    """Extract nr_viagem from annotations by finding and removing 'stcp:nr_viagem:' prefix"""
    if not isinstance(annotations, list):
        return None
    for ann in annotations:
        if isinstance(ann, str) and ann.startswith('stcp:nr_viagem:'):
            return ann.replace('stcp:nr_viagem:', '')
    return None


def save_realtime_data_to_csv(entities, db_dir='db'):
    """Save real-time bus location data to CSV file organized by date"""
    # Ensure db directory exists
    if not os.path.exists(db_dir):
        os.makedirs(db_dir)
    
    # Get today's date in yyyymmdd format
    today = datetime.now().strftime('%Y%m%d')
    csv_path = os.path.join(db_dir, f'{today}.csv')
    
    # Check if file exists to determine if we need to write headers
    file_exists = os.path.exists(csv_path)
    
    # Prepare rows to write
    rows = []
    for e in entities:
        ann = e.get("annotations", {}).get("value", [])
        if not isinstance(ann, list):
            continue
        
        # Check if entity has required fields
        route_found = next((a for a in ann if a in NEEDLE_ROUTES), None)
        if route_found is None:
            continue
        else:
            route_found = route_found.split(':')[-1]  # Extract just the line number for CSV
        sentido_found = next((s for s in ann if s in NEEDLE_SENTIDOS), None)
        if sentido_found is None:
            continue
        else:
            sentido_found = sentido_found.split(':')[-1]  # Extract just the sentido number for CSV
        
        coords = e.get("location", {}).get("value", {}).get("coordinates")
        if not (isinstance(coords, list) and len(coords) >= 2):
            continue
        
        lon, lat = coords[0], coords[1]
        timestamp = format_time(get_timestamp(e))
        vehicle_id = extract_vehicle_id(e.get("id"))
        nr_viagem = get_nr_viagem(ann)
        
        rows.append({
            'veiculo': vehicle_id,
            'hora': timestamp,
            'latitude': lat,
            'longitude': lon,
            'linha': route_found,
            'sentido': sentido_found,
            'nr_viagem': nr_viagem
        })
    
    # Write to CSV
    if rows:
        fieldnames = ['veiculo', 'hora', 'latitude', 'longitude', 'linha', 'sentido', 'nr_viagem']
        with open(csv_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerows(rows)
        
        print(f"Saved {len(rows)} records to {csv_path}")
    
    return len(rows)


def filter_latest_points(entities):
    points = []
    # determine which annotations to match: UI selection if present, else defaults
    annotations_to_check = set(globals().get('SELECTED_ANNOTATIONS', set(NEEDLE_ROUTES)))

    for e in entities:
        ann = e.get("annotations", {}).get("value", [])
        # annotations must be a list
        if not isinstance(ann, list):
            continue
        # require the entity to have one of the selected route annotations
        route_found = next((a for a in ann if a in annotations_to_check), None)
        if route_found is None:
            continue
        # find which sentido (direction) is present
        sentido_found = next((s for s in ann if s in NEEDLE_SENTIDOS), None)
        if sentido_found is None:
            continue

        coords = e.get("location", {}).get("value", {}).get("coordinates")
        if not (isinstance(coords, list) and len(coords) >= 2):
            continue

        lon, lat = coords[0], coords[1]
        heading_val = get_heading(e)
        try:
            heading_val = float(heading_val) if heading_val is not None else None
        except Exception:
            heading_val = None

        try:
            route_line_int = int(route_found.split(':')[-1])
        except Exception:
            route_line_int = None

        points.append({
            "id": e.get("id"),
            "entity_timestamp": get_timestamp(e),
            "lat": lat,
            "lon": lon,
            "sentido": sentido_found,
            "heading": heading_val,
            "route": route_found,
            "route_line": route_line_int,
        })
    return points


def load_routes_gtfs(path="routes.txt"):
    opts = []
    id_to_label = {}
    if not os.path.exists(path):
        return [], {}
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            route_id = row.get('route_id') or ''
            short = row.get('route_short_name') or ''
            long = row.get('route_long_name') or ''
            bgcolor = color_for_route_and_sentido(str(route_id), "stcp:sentido:1", "background")
            txtcolor = color_for_route_and_sentido(str(route_id), "stcp:sentido:1", "text") 
            html_label = f"<span style='background-color:{bgcolor}; color:{txtcolor}; vertical-align:middle;'><b> {short} </b></span>&nbsp{long}"
            opts.append((route_id, html_label))
            id_to_label[route_id] = html_label
    return opts, id_to_label


def write_map(points, history=None, refresh_seconds=30, gtfs_path='routes.txt'):

    routes_opts, route_labels = load_routes_gtfs(gtfs_path)
    
    # Get current time for delay calculation
    now = datetime.now(timezone.utc)
    
    # Calculate real_time_delay using the first point's timestamp (assume all are the same)
    real_time_delay = None
    display_ts = None
    if points:
        first_point_ts = points[0].get("entity_timestamp")
        first_dt = parse_time_to_dt(first_point_ts)
        if first_dt is not None:
            # Ensure timezone awareness
            if first_dt.tzinfo is None:
                first_dt = first_dt.replace(tzinfo=timezone.utc)
            real_time_delay = (now - first_dt).total_seconds()
            display_ts = first_dt.strftime('%H:%M:%S')

    # Center map
    if points:
        center = [points[0]["lat"], points[0]["lon"]]
        zoom = 13
    else:
        center = [41.1579, -8.6291]  # Porto fallback
        zoom = 12

    m = folium.Map(location=center, zoom_start=zoom)
    fetched_at = now.isoformat(timespec="seconds")

    # # Draw previous sets as small circles (history may be None)
    # if history:
    #     # draw oldest first so newer small circles appear on top
    #     for prev_set in reversed(history):
    #         for p in prev_set:
    #             sentido = p.get("sentido")
    #             rl = str(p.get('route_line'))
    #             bg_color = color_for_route_and_sentido(rl, sentido, "background")
    #             folium.CircleMarker(
    #                 location=[p["lat"], p["lon"]],
    #                 radius=4,
    #                 color=bg_color,
    #                 fill=True,
    #                 fill_color=bg_color,
    #                 fill_opacity=0.9,
    #                 weight=1,
    #             ).add_to(m)
    
    # Draw previous sets as small circles (history may be None)
    if history:
        # draw oldest first so newer small circles appear on top
        for prev_set in reversed(history):
            for p in prev_set:
                hora = format_time(p.get("entity_timestamp"))
                sentido = p.get("sentido")
                rl = str(p.get('route_line'))
                bg_color = color_for_route_and_sentido(rl, sentido, "background")
                
                # derive route_id for filtering
                route_id_str = str(rl) if rl is not None else (p.get('route').split(':')[-1] if p.get('route') else '')
                # create small circle using DivIcon with route-marker wrapper
                circle_html = f"<div class='route-marker' data-route='{route_id_str}' style='width:8px;height:8px;border-radius:50%;background:{bg_color};border:1px solid rgba(0,0,0,0.3);'></div>"
                
                popup = (
                    f"Hora: {hora}<br>"
                    f"Veículo: {p['id'].split(':')[-1]}<br>"
                )

                folium.Marker(
                    location=[p["lat"], p["lon"]],
                    icon=DivIcon(html=circle_html, icon_size=(8,8)),
                    popup=popup
                ).add_to(m)

    # Draw current points as rectangular text boxes (DivIcon) with an arrow glyph
    for p in points:
        hora = format_time(p.get("entity_timestamp"))
        popup = (
            f"Hora: {hora}<br>"
            f"Veículo: {p['id'].split(':')[-1]}<br>"
            f"Sentido: {p.get('sentido').split(':')[-1].replace('0', 'Ida').replace('1', 'Volta') if p.get('sentido') else 'N/A'}<br>"
            f"Linha: {p.get('route_line') if p.get('route_line') is not None else p.get('route')}"
        )
        sentido = p.get("sentido")
        rl = str(p.get('route_line'))
        bg_color = color_for_route_and_sentido(rl, sentido, "background")

        # Determine arrow glyph and placement relative to the line number
        glyph = ''
        h = p.get('heading')
        if h is not None:
            try:
                dir_code = heading_to_direction(float(h))
                glyph = direction_glyph(dir_code, use_ascii=USE_ASCII_ARROWS)
            except Exception:
                glyph = ''

        # position arrow before or after: before if heading in [180,360), after if [0,180)
        arrow_html = f"<span style='margin:0 6px;color:#fff;font-weight:700;'>{glyph}</span>" if glyph else ''
        rl_display = str(rl) if rl is not None else (p.get('route') or '')
        if h is not None and 0 <= float(h) < 180:
            label_html = f"<span style='display:inline-block;border-radius:4px;padding:1px 1px;background:{bg_color};color:#ffffff;border:1px solid rgba(0,0,0,0.15);font-weight:700;font-family:Arial,Helvetica,sans-serif'>&nbsp{rl_display}{arrow_html}</span>"
        else:
            label_html = f"<span style='display:inline-block;border-radius:4px;padding:1px 1px;background:{bg_color};color:#ffffff;border:1px solid rgba(0,0,0,0.15);font-weight:700;font-family:Arial,Helvetica,sans-serif'>{arrow_html}{rl_display}&nbsp</span>"

        # folium.Marker(
        #     [p["lat"], p["lon"]],
        #     popup=popup,
        #     icon=DivIcon(html=label_html, icon_size=(70,24))
        # ).add_to(m)

        # derive route id string to match GTFS `route_id` values used in the selector
        route_id_str = str(rl) if rl is not None else (p.get('route').split(':')[-1] if p.get('route') else '')
        wrapped = f"<div class='route-marker' data-route='{route_id_str}'>" + label_html + "</div>"
        folium.Marker(
            [p["lat"], p["lon"]],
            popup=popup,
            icon=DivIcon(html=wrapped, icon_size=(70,24))
        ).add_to(m)

    # Show last-seen boxes for buses missing in current refresh (within history)
    if history:
        current_ids = set(x['id'] for x in points)
        missing = {}
        # history is ordered newest-first (history[0] most recent past), so iterate to capture most recent datapoint
        for hist_set in history:
            for hp in hist_set:
                hid = hp.get('id')
                if hid and hid not in current_ids and hid not in missing:
                    missing[hid] = hp

        for hp in missing.values():
            hora = format_time(hp.get("entity_timestamp"))
            popup = (
                f"Hora: {hora}<br>"
                f"Veículo: {hp['id'].split(':')[-1]}<br>"
                f"Sentido: {hp.get('sentido').replace('0', 'Ida').replace('1', 'Volta') if hp.get('sentido') else 'N/A'}<br>"
                f"Linha: {hp.get('route_line') if hp.get('route_line') is not None else hp.get('route')}"
            )
            # Gray box with line number and arrow for last-seen position
            # Determine arrow glyph and placement relative to the line number
            glyph = ''
            h = hp.get('heading')
            if h is not None:
                try:
                    dir_code = heading_to_direction(float(h))
                    glyph = direction_glyph(dir_code, use_ascii=USE_ASCII_ARROWS)
                except Exception:
                    glyph = ''

            # position arrow before or after: before if heading in [180,360), after if [0,180)
            arrow_html = f"<span style='margin:0 6px;color:#fff;font-weight:700;'>{glyph}</span>" if glyph else ''
            rl_display = str(hp.get('route_line')) if hp.get('route_line') is not None else (hp.get('route') or '')
            if h is not None and 0 <= float(h) < 180:
                label_html = f"<span style='display:inline-block;border-radius:4px;padding:1px 1px;background:#AAAAAA;color:#ffffff;border:1px solid rgba(0,0,0,0.15);font-weight:700;font-family:Arial,Helvetica,sans-serif'>&nbsp{rl_display}{arrow_html}</span>"
            else:
                label_html = f"<span style='display:inline-block;border-radius:4px;padding:1px 1px;background:#AAAAAA;color:#ffffff;border:1px solid rgba(0,0,0,0.15);font-weight:700;font-family:Arial,Helvetica,sans-serif'>{arrow_html}{rl_display}&nbsp</span>"

            # rl_display = str(hp.get('route_line')) if hp.get('route_line') is not None else (hp.get('route') or '')
            # label_html = f"<span style='display:inline-block;border-radius:4px;padding:1px 1px;background:#AAAAAA;color:#ffffff;border:1px solid rgba(0,0,0,0.15);font-weight:700;font-family:Arial,Helvetica,sans-serif'>{rl_display}&nbsp</span>"
            folium.Marker(
                [hp["lat"], hp["lon"]],
                popup=popup,
                icon=DivIcon(html=label_html, icon_size=(70,24)),
            ).add_to(m)

    # Build in-map selector HTML (checkbox list) from GTFS
    options_html = ''
    for rid, route_label in routes_opts:
        options_html += f"""<label style='display:block; cursor:pointer;'>&nbsp;<input type='checkbox' class='route-checkbox' value='{rid}' checked style='vertical-align:middle;'>&nbsp{route_label}</label>\n"""
        # bgcolor = color_for_route_and_sentido(str(rid), "stcp:sentido:1", "background")
        # txtcolor = color_for_route_and_sentido(str(rid), "stcp:sentido:1", "text") 
        # options_html += f"""
        # <label style='display:block; cursor:pointer;'>
        #     &nbsp;<input type='checkbox' class='route-checkbox' value='{rid}' checked style='vertical-align:middle;'> 
        #     <span style='background-color:{bgcolor}; color:{txtcolor}; vertical-align:middle;'>{route_label}</span>
        # </label>\n"""

    control_html = f"""
    <div id='route-filter' style='position:absolute;top:8px;right:8px;z-index:1000;background:rgba(255,255,255,0.95);padding:0 0 0 6px;border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,0.3);font-family:Arial,Helvetica,sans-serif;max-height:160px;overflow-y:auto;min-width:220px;'>
        <div style='position:sticky;top:0;z-index:1001;background:rgba(255,255,255,0.95);margin:0;padding:6px;display:flex;align-items:center;justify-content:space-between;border-bottom:1px solid #ddd;'>
            <div style='font-weight:700;'>Escolha as Linhas</div>
            <div>
                <button id='routeClear'>Mostrar todas</button>
                <button id='routeUncheck' style='margin-left:6px;'>Limpar</button>
            </div>
        </div>
        {options_html}
    </div>
    """

    # JS to filter markers using checkboxes
    script = """
    <script>
    function updateMarkerVisibility() {
        const boxes = document.querySelectorAll('.route-checkbox');
        const selected = Array.from(boxes).filter(b=>b.checked).map(b=>b.value);
        const markers = document.querySelectorAll('.route-marker');
        markers.forEach(function(m){
            const route = m.getAttribute('data-route');
            let container = m.closest('.leaflet-marker-icon');
            if(!container) container = m.parentElement;
            if(selected.length === 0) {
                // when nothing selected, hide all markers ("Limpar")
                if(container) container.style.display = 'none';
            } else {
                if(selected.indexOf(route) !== -1) {
                    if(container) container.style.display = '';
                } else {
                    if(container) container.style.display = 'none';
                }
            }
        });
    }
    document.addEventListener('DOMContentLoaded', function(){
        const boxes = document.querySelectorAll('.route-checkbox');
        boxes.forEach(b=>b.addEventListener('change', updateMarkerVisibility));
        const clearBtn = document.getElementById('routeClear');
        if(clearBtn) clearBtn.addEventListener('click', function(){
            const bs = document.querySelectorAll('.route-checkbox');
            bs.forEach(b=>b.checked=true);
            updateMarkerVisibility();
        });
        const uncheckBtn = document.getElementById('routeUncheck');
        if(uncheckBtn) uncheckBtn.addEventListener('click', function(){
            const bs = document.querySelectorAll('.route-checkbox');
            bs.forEach(b=>b.checked=false);
            updateMarkerVisibility();
        });
        // initial visibility
        updateMarkerVisibility();
    });
    </script>
    """

    # Insert control and script into map
    m.get_root().html.add_child(folium.Element(control_html + script))

    # ✅ Auto-refresh the HTML when opened in a browser tab
    m.get_root().html.add_child(
        folium.Element(f"<meta http-equiv='refresh' content='{refresh_seconds}'>")
    )

    # Insert real-time status info box at bottom
    if display_ts is not None and real_time_delay is not None:
        if real_time_delay < 70:
            bg_color = '#5FDDC2'
            text_color = 'white'
            # text_color = '#0074C7'
            msg = f"Localizações atualizadas! ({display_ts})"
        elif real_time_delay < 180:
            bg_color = 'yellow'
            text_color = 'black'
            msg = f"Localizações com ligeiro atraso ({display_ts})"
        else:
            bg_color = 'red'
            text_color = 'white'
            msg = f"Localizações desatualizadas! ({display_ts})"
        
        info_box_html = f"""
        <div id='real_time_info' style='position:absolute; bottom:40px; right:8px; z-index:1000; background:{bg_color}; color:{text_color}; padding:6px 10px; border-radius:4px; font-family:Arial,Helvetica,sans-serif; font-weight:700; font-size:14px;'>
            {msg}
        </div>
        """
        m.get_root().html.add_child(folium.Element(info_box_html))

    m.save(OUT_MAP)
    return len(points), fetched_at

# Cell 4 — run (updates map every 2 minutes + opens it once in your browser)
opened = False

while True:
    started = time.time()
    try:    
        entities = fetch_entities()
        
        # Save real-time data to CSV
        csv_records = save_realtime_data_to_csv(entities)
        
        points = filter_latest_points(entities)
        n, fetched_at = write_map(points, history=history, refresh_seconds=REFRESH_SECONDS, gtfs_path=ROUTES_GTFS_PATH)

        print(f"Updated: {fetched_at} | Markers: {n} | CSV Records: {csv_records} | Wrote: {OUT_MAP}")
        print(points)

        # Open the map once (after the first successful write)
        if not opened:
            webbrowser.open("file://" + os.path.abspath(OUT_MAP))
            opened = True

        # Update history: insert current points at front and keep only last 5 previous refreshes
        history.insert(0, [p.copy() for p in points])
        history = history[:5]

    except Exception as e:
        print(f"{datetime.now().isoformat(timespec='seconds')}: ERROR: {e}")

    time.sleep(max(0, REFRESH_SECONDS - (time.time() - started)))


Saved 37 records to db\20260301.csv
Updated: 2026-03-01T16:01:47+00:00 | Markers: 37 | CSV Records: 37 | Wrote: map.html
[{'id': 'urn:ngsi-ld:Vehicle:porto:stcp:bus:2963', 'entity_timestamp': '2026-03-01T16:00:37.00Z', 'lat': 41.185897827, 'lon': -8.696310997, 'sentido': 'stcp:sentido:1', 'heading': 205.0, 'route': 'stcp:route:500', 'route_line': 500}, {'id': 'urn:ngsi-ld:Vehicle:porto:stcp:bus:3464', 'entity_timestamp': '2026-03-01T16:00:37.00Z', 'lat': 41.180519104, 'lon': -8.645831108, 'sentido': 'stcp:sentido:1', 'heading': 205.0, 'route': 'stcp:route:503', 'route_line': 503}, {'id': 'urn:ngsi-ld:Vehicle:porto:stcp:bus:3228', 'entity_timestamp': '2026-03-01T16:00:37.00Z', 'lat': 41.166412354, 'lon': -8.627366066, 'sentido': 'stcp:sentido:1', 'heading': 249.0, 'route': 'stcp:route:803', 'route_line': 803}, {'id': 'urn:ngsi-ld:Vehicle:porto:stcp:bus:3562', 'entity_timestamp': '2026-03-01T16:00:37.00Z', 'lat': 41.225547791, 'lon': -8.615667343, 'sentido': 'stcp:sentido:0', 'heading': 